# Warm start on the IDAES CSTR

Under closed-loop control the next problem is the last one moved one
step, so the last solution moved one step is nearly its answer.
`drto.warm_start_dynamic` shifts it. This notebook runs one loop
iteration on the pattern a loop actually uses: one persistent scaled
model, built and scaled once, then solved, shifted, and solved again in
its own units, with the solver's warm-start options at the call site.
The shift carries values only; the solver rebuilds its multipliers
from them in the first iteration, faster than any carried certificate
(gh #36). The solver is ipopt, which consumes the options through its
standard interface.

## One model, built and scaled once

Declarations, the multiplier suffixes, the setpoint, the terminal
segment, the cold start, and the assembly, then a single
`core.scale_model` clone that the whole loop keeps.

In [1]:
import contextlib, io, time

import pyomo.environ as pyo

import drto
from models.idaes_cstr import DC_START, F_IN, VOLUME, build, scaled_solve, tag_scaling

m = build()

ss = pyo.TransformationFactory("drto.steady_state_simulation").create_using(
    m, controls={m.fs.cstr.control_volume.heat.name: 0.0,
                 m.fs.cstr.inlet.flow_vol.name: F_IN})
scaled_solve(ss)
cvs = ss.fs.cstr.control_volume
for j, ssp in (("NaOH", m.ss_naoh), ("EthylAcetate", m.ss_ea),
               ("SodiumAcetate", m.ss_sa), ("Ethanol", m.ss_etoh)):
    ssp.set_value(pyo.value(cvs.material_holdup["Liq", j]))
for j, sgn in (("NaOH", 1), ("EthylAcetate", 1),
               ("SodiumAcetate", -1), ("Ethanol", -1)):
    m.mat0[j] = pyo.value(cvs.material_holdup["Liq", j]) + sgn * DC_START * VOLUME
for k in m.eng_ss:
    m.eng_ss[k] = pyo.value(cvs.energy_holdup[k])

pyo.TransformationFactory("drto.infinite_horizon").apply_to(m)
tag_scaling(m)
drto.cold_start_dynamic(m, profile="exponential", time_constant=3.0)
pyo.TransformationFactory("drto.dynamic_optimization").apply_to(m)

sm = pyo.TransformationFactory("core.scale_model").create_using(m, rename=False)
fmap = sm.component_scaling_factor_map
res = pyo.SolverFactory("ipopt").solve(sm, tee=True)
print(res.solver.termination_condition)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

optimal


## One step later: shift everything

The loop implements the first move and the state advances one sample;
the model's own solution at t = h stands in for the measurement (read
back through the scaling factors, since the hooks hold physical
values). The shift moves every variable one sampling time forward.

In [2]:
cv = sm.fs.cstr.control_volume
h = 1.0
for j in ("NaOH", "EthylAcetate", "SodiumAcetate", "Ethanol"):
    vd = cv.material_holdup[h, "Liq", j]
    sm.mat0[j] = pyo.value(vd) / fmap[vd]
vd = cv.energy_holdup[h, "Liq"]
sm.eng0["Liq"] = pyo.value(vd) / fmap[vd]

print(drto.warm_start_dynamic(sm))

drto warm_start_dynamic (the previous solution, one step on)
  shift         : 1 time units
  copied        : 1000 values on aligned points
  interpolated  : 495 values between points
  filled        : 0 values past the end
  tail          : shifted through t = tN + atanh(tau)/gamma


## The second solve, warm

The same model again, the solver told to trust the start: the shifted
values as the initial point, the barrier already small, the bound
pushes tiny so the active set stays put. These are solve-call options,
not part of the shift.

In [3]:
res = pyo.SolverFactory("ipopt").solve(sm, options={
    "warm_start_init_point": "yes",
    "mu_init": 1e-6,
    "warm_start_bound_push": 1e-9,
    "warm_start_mult_bound_push": 1e-9,
}, tee=True)
print(res.solver.termination_condition)

Ipopt 3.13.2: warm_start_init_point=yes
mu_init=1e-06
warm_start_bound_push=1e-09
warm_start_mult_bound_push=1e-09


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
 

optimal


The cold solve above took its twenty-odd iterations; the
warm-started one lands in single digits: the shifted solution is nearly
the answer, and the solve rebuilds its multipliers from it and stops.
That is warm starting a receding horizon, whole.